In [3]:
import numpy as np
import pandas as pd
import glob, os, warnings
warnings.filterwarnings('ignore')

# cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/criticalConcentrations_updated.csv")
cc_df = pd.read_csv("drug_CC.csv")
cc_df_who = pd.read_csv("/home/sak0914/who-analysis/data/drug_CC.csv")

isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")

drug_abbr_dict = {"Delamanid": "DLM",
                  "Bedaquiline": "BDQ",
                  "Clofazimine": "CFZ",
                  "Ethionamide": "ETO",
                  "Linezolid": "LZD",
                  "Moxifloxacin": "MXF",
                  "Capreomycin": "CAP",
                  "Amikacin": "AMK",
                  "Pretomanid": "PMD",
                  "Pyrazinamide": "PZA",
                  "Kanamycin": "KAN",
                  "Levofloxacin": "LFX",
                  "Streptomycin": "STM",
                  "Ethambutol": "EMB",
                  "Isoniazid": "INH",
                  "Rifampicin": "RIF",
                 }

abbr_drug_dict = {val: key for key, val in drug_abbr_dict.items()}

abbr_drug_dict.update({'MFX': 'Moxifloxacin', # 'MB' ????
                       'STR': 'Streptomycin', # 'SIT' ????
                       'OFX': 'Ofloxacin',
                       'CYC': 'Cycloserine',
                       'PAS': 'Paraaminosalicylicacid'
                      })

who_variants = pd.read_csv("../data_processing/data_utils/WHO_catalog_V2.csv", header=[2]).reset_index(drop=True)

df_samples_geno = pd.read_csv("../data_processing/samples_pass_geno_QC.csv")
df_samples_geno['log_F2'] = np.log2(df_samples_geno['F2'])

data_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs"

In [4]:
for drug in os.listdir(data_dir):
    print(f"python3 data_processing/01_combine_MIC_data.py {drug}")
    print(f"python3 data_processing/02_clean_pheno_data.py {drug}")

python3 data_processing/01_combine_MIC_data.py BDQ
python3 data_processing/02_clean_pheno_data.py BDQ
python3 data_processing/01_combine_MIC_data.py DLM
python3 data_processing/02_clean_pheno_data.py DLM
python3 data_processing/01_combine_MIC_data.py EMB
python3 data_processing/02_clean_pheno_data.py EMB
python3 data_processing/01_combine_MIC_data.py ETO
python3 data_processing/02_clean_pheno_data.py ETO
python3 data_processing/01_combine_MIC_data.py INH
python3 data_processing/02_clean_pheno_data.py INH
python3 data_processing/01_combine_MIC_data.py LFX
python3 data_processing/02_clean_pheno_data.py LFX
python3 data_processing/01_combine_MIC_data.py MXF
python3 data_processing/02_clean_pheno_data.py MXF
python3 data_processing/01_combine_MIC_data.py PZA
python3 data_processing/02_clean_pheno_data.py PZA
python3 data_processing/01_combine_MIC_data.py RIF
python3 data_processing/02_clean_pheno_data.py RIF


# Farhat Internal

In [19]:
# df_rollingDB = pd.read_csv("./rollingDB_MIC_raw.csv")

# for col in df_rollingDB.columns:

#     if 'ETH' in col:
#         df_rollingDB.rename(columns={col: col.replace(col, col.replace('ETH', 'ETO'))}, inplace=True)

#     if 'LEV' in col:
#         df_rollingDB.rename(columns={col: col.replace(col, col.replace('LEV', 'LFX'))}, inplace=True)

#     if 'AMI' in col:
#         df_rollingDB.rename(columns={col: col.replace(col, col.replace('AMI', 'AMK'))}, inplace=True)

# df_rollingDB.to_csv("./rollingDB_MIC_raw.csv", index=False)

In [50]:
# 7H9 and 7H10 critical concentrations are the same
# cc_df = pd.read_csv("./drug_CC.csv")
# i = 0
# cc_7H9_add = pd.DataFrame(columns=cc_df.columns)

# for drug in cc_df.query("Medium=='7H10'").Drug.unique():
#     cc_7H10 = cc_df.query("Medium=='7H10' & Drug==@drug")['Value'].values[0]
#     print(drug, cc_7H10)
#     cc_7H9_add.loc[i, :] = ['7H9', cc_7H10, drug]
#     i += 1

# # save
# pd.concat([cc_df, cc_7H9_add]).drop_duplicates().to_csv("./drug_CC.csv")

# CRyPTIC Drug Renaming

In [62]:
# df_cryptic = pd.read_csv("./CRyPTIC_normalized.csv")

# for col in df_cryptic.columns:

#     if 'ETH' in col:
#         df_cryptic.rename(columns={col: col.replace(col, col.replace('ETH', 'ETO'))}, inplace=True)

#     if 'LEV' in col:
#         df_cryptic.rename(columns={col: col.replace(col, col.replace('LEV', 'LFX'))}, inplace=True)

#     if 'AMI' in col:
#         df_cryptic.rename(columns={col: col.replace(col, col.replace('AMI', 'AMK'))}, inplace=True)

# df_cryptic.to_csv("./CRyPTIC_normalized.csv", index=False)

# MIC-ML Consortium

A lot of work to re-normalize everything from source, so just fix Rifampicin because of the updates to the critical concentration.

<ul>
    <li>MGIT: 1 -> 0.5</li>
    <li>7H11: 0.5 -> 1</li>
</ul>

According to the critical concentration updates: https://iris.who.int/bitstream/handle/10665/339275/9789240017283-eng.pdf?sequence=1&isAllowed=y

the RIF CC in 7H11 and LJ were not changed, but they were lowered from 1 to 0.5 µg/mL for MGIT and 7H10. So need to just change MGIT and 7H11 because they were using switched MICs.

Multiply the MGIT MIC by 2 and the 7H11 CC by 0.5.

$MIC_{MGIT} \cdot \frac{CC_{7H10}}{CC_{MGIT}} = MIC_{MGIT} \cdot \frac{0.5}{1} = 0.5 MIC_{MGIT}$

But it should have been

$MIC_{MGIT} \cdot \frac{CC_{7H10}}{CC_{MGIT}} = MIC_{MGIT} \cdot \frac{1}{1} = MIC_{MGIT}$

And for 7H11:

$MIC_{7H11} \cdot \frac{CC_{7H10}}{CC_{7H11}} = MIC_{7H11} \cdot \frac{0.5}{0.5} = MIC_{7H11}$

But it should have been

$MIC_{7H11} \cdot \frac{CC_{7H10}}{CC_{7H11}} = MIC_{7H11} \cdot \frac{0.5}{1} = 0.5 MIC_{7H11}$

In [362]:
df_MIC_ML = pd.read_csv("MIC_ML.csv")

In [355]:
# # fix RIF critical concentrations
# df_MIC_ML.loc[(df_MIC_ML['MEDIA']=='MGIT'), 'RIF'] = df_MIC_ML.loc[(df_MIC_ML['MEDIA']=='MGIT')]['RIF'] * 2
# df_MIC_ML.loc[(df_MIC_ML['MEDIA']=='7H11'), 'RIF'] = df_MIC_ML.loc[(df_MIC_ML['MEDIA']=='7H11')]['RIF'] * 0.5

# # update to public IDs where possible
# new_name_mapping_dict = {}

# for name in df_MIC_ML['ROLLINGDB_ID'].values:

#     if name in isolate_metadata.ROLLINGDB_ID.values:
#         new_name_mapping_dict[name] = name
    
#     elif name in isolate_metadata.OtherNames.values:
#         new_name_mapping_dict[name] = isolate_metadata.query("OtherNames==@name").ROLLINGDB_ID.values[0]
#     # many are not there because we didn't receive FASTQs for them
#     # else:
#     #     print(f"{name} not found")

# print(len(new_name_mapping_dict))

# df_MIC_ML['ROLLINGDB_ID'] = df_MIC_ML['ROLLINGDB_ID'].map(new_name_mapping_dict).fillna(df_MIC_ML['ROLLINGDB_ID'])

# df_MIC_ML.to_csv("MIC_ML.csv", index=False)

2036


In [330]:
sample_id = 'KOR-K2016-0044'

print(f"bcftools filter -i 'POS >= 759807 & POS <= 767320' /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/{sample_id}/WHO_resistance/{sample_id}_variants_combinedCodons.eff.vcf")

bcftools filter -i 'POS >= 759807 & POS <= 767320' /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/KOR-K2016-0044/WHO_resistance/KOR-K2016-0044_variants_combinedCodons.eff.vcf


In [324]:
who_variants.query("variant=='rpoB_p.Leu430Pro'")

,drug,gene,mutation,variant,tier,effect,genomic position,algorithm_pass,Present_SOLO_SR,Present_SOLO_R,...,Additional grading criteria applied,FINAL CONFIDENCE GRADING,Comment,CHANGES vs ver1,"Relaxed thresholds simulation (BDQ_Rv0678, CFZ_Rv0678, INH_katG, DLM_ddn/fbiA/fbiB/fbiC/fgd1/Rv2983)",Silent mutation,Listed in abridged tables,Additional grading,Footnote,CHANGES vs ver1.1
41336,Rifampicin,rpoB,p.Leu430Pro,rpoB_p.Leu430Pro,1,missense_variant,"(see ""Genomic_coordinates"" sheet)",1.0,208.0,53.0,...,Borderline,1) Assoc w R,NaN,No change,NaN,NaN,yes,Borderline,NaN,1


# WHO Catalog MIC Data from Sacha Laurent, Fall 2024

In [378]:
# df_WHO_catalog = pd.read_csv("./2023_WHO_catalog_MIC.csv")
# df_WHO_catalog['Medium'] = df_WHO_catalog['plate'].copy()

# df_WHO_catalog.loc[df_WHO_catalog['Medium'].str.contains('7h10', case=False), 'Medium'] = '7H10'
# df_WHO_catalog.loc[df_WHO_catalog['Medium'].str.contains('UKMYC5', case=False), 'Medium'] = 'UKMYC5'
# df_WHO_catalog.loc[df_WHO_catalog['Medium'].str.contains('microdilution', case=False), 'Medium'] = 'BMD'

# print(df_WHO_catalog.Medium.unique())

# df_WHO_catalog.to_csv("./2023_WHO_catalog_MIC.csv", index=False)

array(['UKMYC5', 'MGIT', 'REMA', 'UKMYC6', 'LJ', '7H11', '7H10',
       'colourmetric', 'MYCOTB', '7H9', 'BMD'], dtype=object)

In [417]:
df_single_drug.query("MEDIA not in ['UKMYC5', 'UKMYC6']")

,ROLLINGDB_ID,MEDIA,ORIG_CC,MEDIA_NORM,MEDIA_NORM_CC,BDQ_lower_bound,BDQ_upper_bound
32,SAMEA11006683,MGIT,1.00,7H11,0.25,0.00000,0.03125
33,SAMEA11006684,MGIT,1.00,7H11,0.25,0.00000,0.03125
109,SAMEA114089356,MGIT,1.00,7H11,0.25,0.00000,0.03125
110,SAMEA114089359,MGIT,1.00,7H11,0.25,0.00000,0.03125
111,SAMEA114089363,MGIT,1.00,7H11,0.25,0.00000,0.03125
...,...,...,...,...,...,...,...
1578,SAMN18498654,BMD,0.12,7H11,0.25,0.06250,0.12500
1579,SAMN18498655,BMD,0.12,7H11,0.25,0.03125,0.06250
1580,SAMN18498656,BMD,0.12,7H11,0.25,0.03125,0.06250
1581,SAMN29500381,7H11,0.25,7H11,0.25,0.25000,0.50000


In [472]:
df_BDQ = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/BDQ/combined_MIC.csv")

In [487]:
df_BDQ.query("ROLLINGDB_ID=='SAMEA8742439'")

,ROLLINGDB_ID,BDQ_PHENOTYPE_QUALITY,BDQ_upper_bound,BDQ_lower_bound,BDQ_midpoint,BDQ_MEDIA_NORM,BDQ_lower_bound_NORM,BDQ_midpoint_NORM,BDQ_upper_bound_NORM,MEDIA,DB_OF_ORIGIN,MEDIA_NORM,MEDIA_NORM_CC
9763,SAMEA8742439,HIGH,0.12,0.06,0.09,7H11,0.06,0.09,0.12,UKMYC,CRyPTIC,NaN,NaN


In [477]:
cryptic_og = pd.read_csv("./CRyPTIC_reuse_table_20231208.csv")

In [488]:
isolate_metadata.query("ROLLINGDB_ID=='SAMEA8742439'")

,ROLLINGDB_ID,BIOSAMPLE_ACCESSION,SAMPLE,RUN,DB_OF_ORIGIN,ISOLATION_DATE,ISOLATION_COUNTRY,ISOLATION_REGION,SPUTUM/CULTURE,UNPAIRED/PAIRED,Platform,Model,SPECIES,TaxID,ReleaseDate,LoadDate,Combined_Runs,Num_Runs,OtherNames
44018,SAMEA8742439,SAMEA8742439,ERS6422020,ERR5918109,cryptic,NaN,NaN,NaN,1,1,ILLUMINA,Illumina HiSeq 4000,Mycobacterium tuberculosis,1773.0,2021-12-02 12:16:18,2021-11-11 13:01:06,ERR5918109,1.0,NaN


In [489]:
cryptic_og.query("ENA_RUN=='ERR5918109'")[cryptic_og.columns[cryptic_og.columns.str.startswith('BDQ')]]

,BDQ_BINARY_PHENOTYPE,BDQ_MIC,BDQ_PHENOTYPE_QUALITY
10892,S,0.12,HIGH


In [474]:
df_BDQ.query("ROLLINGDB_ID=='SAMEA8742282'")

,ROLLINGDB_ID,BDQ_PHENOTYPE_QUALITY,BDQ_upper_bound,BDQ_lower_bound,BDQ_midpoint,BDQ_MEDIA_NORM,BDQ_lower_bound_NORM,BDQ_midpoint_NORM,BDQ_upper_bound_NORM,MEDIA,DB_OF_ORIGIN,MEDIA_NORM,MEDIA_NORM_CC
9659,SAMEA8742282,HIGH,0.008,0.0,0.004,7H11,0.0,0.004,0.008,UKMYC,CRyPTIC,NaN,NaN


In [475]:
cc_df.query("Drug=='Bedaquiline'")

,Medium,Value,Drug
18,UKMYC5,0.25,Bedaquiline
19,UKMYC6,0.25,Bedaquiline
32,7H11,0.25,Bedaquiline
51,MGIT,1.00,Bedaquiline
98,BMD,0.12,Bedaquiline


In [471]:
df_BDQ.query("Span_CC==1")

,ROLLINGDB_ID,BDQ_MEDIA,BDQ_lower_bound,BDQ_midpoint,BDQ_upper_bound,MEDIA,DB_OF_ORIGIN,MEDIA_NORM,MEDIA_NORM_CC,F2,...,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,log2_F2,Span_CC,Binary,Stratify,category
9898,SAMN37124201,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.009391,...,2.2.1.1.1,"beijing,mungi",lin2.2.1,NaN,2,-6.734530,1,NaN,NaN,test_set
9899,SAMN37124218,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.008861,...,4.2.1.2.2.1.1,stype,NaN,4,4,-6.818309,1,NaN,NaN,test_set
9900,SAMN37124224,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.005386,...,4.2.1.2.1.1.i4.1,lam,NaN,4.3/LAM,4,-7.536486,1,NaN,NaN,test_set
9901,SAMN37124231,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.015129,...,4.2.1.2.1.1.i4.1,lam,NaN,4.3/LAM,4,-6.046530,1,NaN,NaN,test_set
9902,SAMN37124228,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.018610,...,4.2.1.2.2.1.1,stype,NaN,4,4,-5.747771,1,NaN,NaN,test_set
9903,SAMN37124261,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.026863,...,4.2.1.2.1.1.i1,lam,NaN,4.3/LAM,4,-5.218227,1,NaN,NaN,test_set
9904,SAMN37124262,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.029978,...,4.2.1.2.1.1.i1,lam,NaN,4.3/LAM,4,-5.059938,1,NaN,NaN,test_set
9905,SAMN37124287,7H11,0.156,0.234,0.312,MGIT,Gandhi,NaN,NaN,0.015826,...,2.2.2,"beijing,ethiopian,mungi","asia_ancestral_1,lin2.2",NaN,2,-5.981514,1,NaN,NaN,test_set
9906,TN8600,7H11,0.160,0.240,0.320,7H10,Kreiswirth,NaN,NaN,0.015173,...,2.2.2,"beijing,mungi","asia_ancestral_1,lin2.2",NaN,2,-6.042359,1,NaN,NaN,test_set
9907,TN15617,7H11,0.160,0.240,0.320,7H10,Kreiswirth,NaN,NaN,0.027660,...,4.1.i1.2.1,xtype,NaN,4,4,-5.176074,1,NaN,NaN,test_set


In [458]:
df_BDQ.query("DB_OF_ORIGIN=='WHO_catalog'").merge(df_samples_geno, on='ROLLINGDB_ID').sort_values("F2", ascending=False)

,ROLLINGDB_ID,BDQ_PHENOTYPE_QUALITY,BDQ_upper_bound,BDQ_lower_bound,BDQ_midpoint,MEDIA_NORM,BDQ_lower_bound_NORM,BDQ_midpoint_NORM,BDQ_upper_bound_NORM,MEDIA,...,MEDIA_NORM_CC,DB_OF_ORIGIN_y,F2,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,log_F2
38,SAMEA114089533,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.109918,2.2.1.1,2.2.1.1.1,beijing,"lin2.2.1,pacific_RD150,asian_african_2",NaN,2,-3.185497
116,SAMN18498628,NaN,NaN,NaN,NaN,NaN,0.125000,NaN,0.250000,BMD,...,0.25,NaN,0.033049,3,3.1.1,"east_african_indian,ghana",NaN,NaN,3,-4.919240
105,SAMN18498614,NaN,NaN,NaN,NaN,NaN,0.520833,NaN,1.041667,BMD,...,0.25,NaN,0.030920,1.2.2,1.2.2.1,"indo_oceanic,ghana,mungi",NaN,NaN,1,-5.015300
41,SAMEA114089556,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.028710,4.2.1,4.2.2.2.2,"ural,canetti",NaN,4,4,-5.122319
80,SAMEA114089861,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.026848,4.2.1,4.2.2.2.1,ural,NaN,4,4,-5.219038
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,SAMEA114089472,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.006355,4.8,4.2.1.1.1.1.1.1.i1,NaN,NaN,4.10/PGG3,4,-7.297826
53,SAMEA114089683,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.005040,4.3.4.2,4.2.1.2.1.1.i3.1,"lam,ural",NaN,4.3/LAM,4,-7.632414
58,SAMEA114089734,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.004510,4.5,4.2.1.1.2,canetti,NaN,4.5,4,-7.792720
52,SAMEA114089637,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.031250,MGIT,...,0.25,NaN,0.004284,4.3.4.2.1,4.2.1.2.1.1.i3.1,"lam,caprae",NaN,4.3/LAM,4,-7.866703


In [455]:
isolate_metadata.query("ROLLINGDB_ID=='SAMEA11006683'")

,ROLLINGDB_ID,BIOSAMPLE_ACCESSION,SAMPLE,RUN,DB_OF_ORIGIN,ISOLATION_DATE,ISOLATION_COUNTRY,ISOLATION_REGION,SPUTUM/CULTURE,UNPAIRED/PAIRED,Platform,Model,SPECIES,TaxID,ReleaseDate,LoadDate,Combined_Runs,Num_Runs,OtherNames
76402,SAMEA11006683,SAMEA11006683,ERS8656462,ERR7362019,NaN,NaN,NaN,NaN,1,1,ILLUMINA,Illumina HiSeq 2500,Mycobacterium tuberculosis,1773.0,2022-01-22 02:42:21,2022-07-28 01:37:47,ERR7362019,1.0,NaN


In [456]:
df_samples_geno.query("ROLLINGDB_ID=='SAMEA11006683'")

,ROLLINGDB_ID,DB_OF_ORIGIN,F2,Coll2014,Freschi2020,Lipworth2019,Shitikov2017,Stucki2016,Lineage,log_F2
15630,SAMEA11006683,NaN,0.003811,2.2.1,2.2.1.1.1,"beijing,mungi",lin2.2.1,NaN,2,-8.035563


In [445]:
df_BDQ.loc[~pd.isnull(df_BDQ['MEDIA_NORM.1'])].DB_OF_ORIGIN.unique()

array(['WHO_catalog'], dtype=object)

In [446]:
df_samples_geno

NameError: name 'df_samples_geno' is not defined

In [436]:
def get_WHO_single_drug_MIC_normalized(df, drug_code):
        
    df_single_drug = df.query("drug_code==@drug_code")

    drug = abbr_drug_dict[drug_code]
    keep_media = cc_df.query("Drug == @drug").Medium.unique()
    df_single_drug = df_single_drug.query("Medium in @keep_media").reset_index(drop=True)[['name', 'range', 'Medium']]

    # get a dictionary to map media to critical concentrations
    single_drug_media_dict = dict(zip(cc_df.query("Drug == @drug")['Medium'], cc_df.query("Drug == @drug")['Value']))

    if drug in ['Bedaquiline', 'Delamanid']:
        media_norm = '7H11'
    elif drug == 'Pyrazinamide':
        media_norm = 'MGIT'
    else:
        media_norm = '7H10'

    df_single_drug['MEDIA_NORM'] = media_norm
    df_single_drug['MEDIA_NORM_CC'] = cc_df.query("Drug==@drug & Medium==@media_norm").Value.values[0]
    df_single_drug['ORIG_CC'] = df_single_drug['Medium'].map(single_drug_media_dict)

    for i, row in df_single_drug.iterrows():
        df_single_drug.loc[i, [f"{drug_code}_lower_bound", f"{drug_code}_upper_bound"]] = row['range'].replace('[', '').replace('(', '').replace(']', '').replace(')', '').split(',')

    # left- and right-censoring will be missing values, replaced with the empty string
    df_single_drug[f"{drug_code}_lower_bound"] = df_single_drug[f"{drug_code}_lower_bound"].replace('', 0)
    df_single_drug[f"{drug_code}_upper_bound"] = df_single_drug[f"{drug_code}_upper_bound"].replace('', np.inf)

    return df_single_drug
    
    # df_single_drug[[f"{drug_code}_lower_bound", f"{drug_code}_upper_bound"]] = df_single_drug[[f"{drug_code}_lower_bound", f"{drug_code}_upper_bound"]].astype(float)
    
    # # multiply by the ratio of MEDIA_NORM_CC to MEDIA_CC to get the MIC in MEDIA_NORM
    # df_single_drug[f"{drug_code}_lower_bound_NORM"] = df_single_drug[f"{drug_code}_lower_bound"] * df_single_drug['MEDIA_NORM_CC'] / df_single_drug['ORIG_CC']
    # df_single_drug[f"{drug_code}_upper_bound_NORM"] = df_single_drug[f"{drug_code}_upper_bound"] * df_single_drug['MEDIA_NORM_CC'] / df_single_drug['ORIG_CC']

    # df_single_drug.rename(columns={'name': 'ROLLINGDB_ID', 'Medium': 'MEDIA'}, inplace=True)
    # df_single_drug['DB_OF_ORIGIN'] = 'WHO_catalog'
    
    # return df_single_drug.query(f"{drug_code}_lower_bound_NORM != {drug_code}_upper_bound_NORM").reset_index(drop=True)[['ROLLINGDB_ID', 'MEDIA', 'ORIG_CC', 'MEDIA_NORM', 'MEDIA_NORM_CC', f"{drug_code}_lower_bound_NORM", f"{drug_code}_upper_bound_NORM"]]

In [439]:
df_single_drug.query(f"BDQ_upper_bound==''")

,name,range,Medium,MEDIA_NORM,MEDIA_NORM_CC,ORIG_CC,BDQ_lower_bound,BDQ_upper_bound
1918,SAMEA7524713,"(2,)",UKMYC5,7H11,0.25,0.25,inf,
2153,SAMEA7525181,"(2,)",UKMYC5,7H11,0.25,0.25,inf,
3188,SAMEA7541778,"(2,)",UKMYC5,7H11,0.25,0.25,inf,
3320,SAMEA7542094,"(2,)",UKMYC5,7H11,0.25,0.25,inf,
4283,SAMEA7545482,"(1,)",UKMYC6,7H11,0.25,0.25,inf,
5388,SAMEA7547547,"(2,)",UKMYC5,7H11,0.25,0.25,inf,
7200,SAMEA7563196,"(1,)",UKMYC6,7H11,0.25,0.25,inf,
7213,SAMEA7563215,"(1,)",UKMYC6,7H11,0.25,0.25,inf,
7486,SAMEA7563608,"(1,)",UKMYC6,7H11,0.25,0.25,inf,


In [437]:
df_single_drug = get_WHO_single_drug_MIC_normalized(df_WHO_catalog, 'BDQ')

In [ ]:
df_single_drug

In [193]:
df_test = get_WHO_single_drug_MIC_normalized(df_WHO_catalog, 'Bedaquiline')

In [149]:
df_WHO_catalog.query("drug_code=='BDQ'").Medium.unique()

array(['UKMYC5', 'MGIT', 'UKMYC6', '7H9', 'BMD', '7H10', '7H11'],
      dtype=object)

In [150]:
df_test.query("name=='SAMEA11006587'")

,name,range,Medium,BDQ_lower_bound,BDQ_upper_bound
421,SAMEA11006587,"[0.5,0.5]",MGIT,0.5,0.5


In [151]:
df_test.query("BDQ_lower_bound != BDQ_upper_bound").Medium.unique()

array(['UKMYC5', 'MGIT', 'UKMYC6', 'BMD', '7H11'], dtype=object)

In [198]:
isolate_metadata.query("OtherNames=='DA301'")

,ROLLINGDB_ID,BIOSAMPLE_ACCESSION,SAMPLE,RUN,DB_OF_ORIGIN,ISOLATION_DATE,ISOLATION_COUNTRY,ISOLATION_REGION,SPUTUM/CULTURE,UNPAIRED/PAIRED,Platform,Model,SPECIES,TaxID,ReleaseDate,LoadDate,Combined_Runs,Num_Runs,OtherNames
1075,SAMN25078201,SAMN25078201,SRS11705217,SRR17657994,NaN,NaN,NaN,NaN,1,1,ILLUMINA,NextSeq 550,Mycobacterium tuberculosis,1773.0,2023-05-01 00:02:05,2022-01-18 19:24:35,SRR17657994,1.0,DA301


In [208]:
isolate_metadata.query("ROLLINGDB_ID.str.contains('B18', case=False)")

,ROLLINGDB_ID,BIOSAMPLE_ACCESSION,SAMPLE,RUN,DB_OF_ORIGIN,ISOLATION_DATE,ISOLATION_COUNTRY,ISOLATION_REGION,SPUTUM/CULTURE,UNPAIRED/PAIRED,Platform,Model,SPECIES,TaxID,ReleaseDate,LoadDate,Combined_Runs,Num_Runs,OtherNames


In [205]:
# isolate_metadata.dropna(subset='OtherNames').query("OtherNames.str.startswith('Erdman', case=False)")

# Data from David Alland, July 2023

In [235]:
df_Alland = pd.read_excel("Alland_Courtney_MICs.xlsx", sheet_name=None)
print(df_Alland.keys())

H37Rv_standards = df_Alland['H37Rv_standards_MGIT']
df_Alland = df_Alland['Fold_change_H37Rv']

# rename TDR samples for consistency
df_Alland['Strain'] = df_Alland['Strain'].str.replace('TDR', 'TDR_')

# keep only those that we have FASTQs for
df_Alland = df_Alland.merge(isolate_metadata[['ROLLINGDB_ID', 'OtherNames']], left_on='Strain', right_on='OtherNames').reset_index(drop=True)

df_Alland['Medium'] = 'MGIT'

dict_keys(['Fold_change_H37Rv', 'H37Rv_standards_MGIT'])


In [296]:
H37Rv_standards

,Drug,H37Rv_MIC_uM,Drug_weight_Da,H37Rv_MIC_ug_mL
0,INH,0.1953,137,0.026756
1,RIF,0.0117,823,0.009629
2,EMB,3.1300,204,0.638520
3,MXF,0.0781,401,0.031318
4,ETO,6.0000,166,0.996000
5,AMK,0.4000,586,0.234400
6,PMD,0.2000,359,0.071800
7,BDQ,0.1000,556,0.055600
8,LZD,0.9760,337,0.328912
9,SQ109,0.7000,331,0.231700


In [297]:
0.0117*823

9.629100000000001

In [293]:
df_test = get_Alland_single_drug_MIC_normalized(df_Alland, 'Rifampicin')

In [299]:
df_test.sort_values("RIF_lower_bound").drop_duplicates('RIF_lower_bound')

,ROLLINGDB_ID,Medium,RIF,MEDIA_NORM,MEDIA_NORM_CC,MEDIA_CC,RIF_LB_orig,RIF_UB_orig,RIF_lower_bound,RIF_upper_bound
28,SAMEA5282528,MGIT,≤0.25,7H10,0.5,0.5,0.0000,0.0025,0.0000,0.0025
29,SAMEA5282533,MGIT,0.5,7H10,0.5,0.5,0.0025,0.0050,0.0025,0.0050
43,SAMEA5282333,MGIT,1,7H10,0.5,0.5,0.0050,0.0100,0.0050,0.0100
4,SAMEA5282422,MGIT,2,7H10,0.5,0.5,0.0100,0.0200,0.0100,0.0200
33,SAMEA5282541,MGIT,4,7H10,0.5,0.5,0.0200,0.0400,0.0200,0.0400
58,SAMEA5282379,MGIT,≥8,7H10,0.5,0.5,0.0800,inf,0.0800,inf


In [ ]:
bcftools filter -i "POS >= 759807 & POS <= 767320" /n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMEA5282541/WHO_resistance/SAMEA5282541_variants_combinedCodons.eff.vcf

In [295]:
cc_df.query("Drug=='Rifampicin'")

,Medium,Value,Drug
2,UKMYC5,0.500,Rifampicin
3,UKMYC6,0.500,Rifampicin
59,7H10,0.500,Rifampicin
61,LJ,40.000,Rifampicin
63,MGIT,0.500,Rifampicin
68,7H11,1.000,Rifampicin
77,MYCOTB,0.500,Rifampicin
96,BMD,0.125,Rifampicin
105,NSW_sensititre_plate,0.500,Rifampicin


In [290]:
who_variants.query("variant in ['mmpS5_p.Val55Met', 'mmpL5_p.Ile948Val', 'mmpL5_p.Thr794Ile'] & drug=='Bedaquiline'")

,drug,gene,mutation,variant,tier,effect,genomic position,algorithm_pass,Present_SOLO_SR,Present_SOLO_R,...,Additional grading criteria applied,FINAL CONFIDENCE GRADING,Comment,CHANGES vs ver1,"Relaxed thresholds simulation (BDQ_Rv0678, CFZ_Rv0678, INH_katG, DLM_ddn/fbiA/fbiB/fbiC/fgd1/Rv2983)",Silent mutation,Listed in abridged tables,Additional grading,Footnote,CHANGES vs ver1.1
2587,Bedaquiline,mmpL5,p.Ile948Val,mmpL5_p.Ile948Val,1,missense_variant,"(see ""Genomic_coordinates"" sheet)",1.0,5468.0,55.0,...,Literature evidence (PMID 28031270; 34503982),4) Not assoc w R - Interim,NaN,DOWN from NotAwR to NotAwRI,NaN,NaN,yes,Lit. (PMID 28031270; 34503982),NaN,3
2668,Bedaquiline,mmpL5,p.Thr794Ile,mmpL5_p.Thr794Ile,1,missense_variant,"(see ""Genomic_coordinates"" sheet)",1.0,4.0,0.0,...,Literature evidence (PMID 28031270; 34503982),4) Not assoc w R - Interim,NaN,DOWN from NotAwR to NotAwRI,NaN,NaN,yes,Lit. (PMID 28031270; 34503982),NaN,3
2798,Bedaquiline,mmpS5,p.Val55Met,mmpS5_p.Val55Met,1,missense_variant,"(see ""Genomic_coordinates"" sheet)",1.0,2.0,0.0,...,NaN,3) Uncertain significance,NaN,No change,NaN,NaN,no,NaN,NaN,1


In [292]:
def get_Alland_single_drug_MIC_normalized(df, drug):

    drug_code = drug_abbr_dict[drug]

    # these were the fold changes relative to the H37Rv MIC that they tested
    testing_fold_changes = [0.25, 0.5, 1, 2, 4, 8]

    # get the H37Rv MIC, which is in MGIT media
    drug_h37Rv_MIC = np.round(H37Rv_standards.query("Drug==@drug_code")['H37Rv_MIC_ug_mL'].values[0], 3)
    
    df_single_drug = df[['ROLLINGDB_ID', 'Medium', drug_code]]

    # get a dictionary to map media to critical concentrations
    single_drug_media_dict = dict(zip(cc_df.query("Drug == @drug")['Medium'], cc_df.query("Drug == @drug")['Value']))

    if drug in ['Bedaquiline', 'Delamanid']:
        media_norm = '7H11'
    else:
        media_norm = '7H10'

    df_single_drug['MEDIA_NORM'] = media_norm
    df_single_drug['MEDIA_NORM_CC'] = cc_df.query("Drug==@drug & Medium==@media_norm").Value.values[0]
    df_single_drug['MEDIA_CC'] = df_single_drug['Medium'].map(single_drug_media_dict)

    for i, row in df_single_drug.iterrows():
        if '≤' in str(row[drug_code]):
            df_single_drug.loc[i, [f"{drug_code}_LB_orig", f"{drug_code}_UB_orig"]] = [0, float(row[drug_code].split('≤')[-1])]
        elif '>' in str(row[drug_code]):
            df_single_drug.loc[i, [f"{drug_code}_LB_orig", f"{drug_code}_UB_orig"]] = [float(row[drug_code].split('>')[-1]), np.inf]
        elif '≥' in str(row[drug_code]):
            df_single_drug.loc[i, [f"{drug_code}_LB_orig", f"{drug_code}_UB_orig"]] = [float(row[drug_code].split('≥')[-1]), np.inf]
        # the upper bound is the fold change and the lower bound is one below
        else:
            bound_idx = testing_fold_changes.index(row[drug_code])
            assert bound_idx != 0 # shouldn't be 0 because that would be the ≤ case above
            df_single_drug.loc[i, [f"{drug_code}_LB_orig", f"{drug_code}_UB_orig"]] = [testing_fold_changes[bound_idx-1], row[drug_code]]

    # multiply by the H37Rv MIC to convert the fold changes to MIC units, in MGIT
    df_single_drug[[f"{drug_code}_LB_orig", f"{drug_code}_UB_orig"]] *= drug_h37Rv_MIC
    
    # multiply by the ratio of MEDIA_NORM_CC to MEDIA_CC to get the MIC in MEDIA_NORM
    df_single_drug[f"{drug_code}_lower_bound"] = df_single_drug[f"{drug_code}_LB_orig"] * df_single_drug['MEDIA_NORM_CC'] / df_single_drug['MEDIA_CC']
    df_single_drug[f"{drug_code}_upper_bound"] = df_single_drug[f"{drug_code}_UB_orig"] * df_single_drug['MEDIA_NORM_CC'] / df_single_drug['MEDIA_CC']
    
    return df_single_drug.query(f"{drug_code}_lower_bound != {drug_code}_upper_bound").reset_index(drop=True)

In [94]:
cc_df_who_final.query("Medium=='7H9'").sort_values("Drug")

,Medium,Value,Drug


In [33]:
cc_df_who.Medium.unique()

array(['UKMYC5', 'UKMYC6', 'MGIT', '7H10', 'LJ', '7H11', 'MYCOTB', 'BMD'],
      dtype=object)

In [81]:
cc_df_who.query("Drug=='Delamanid'")

,Medium,Value,Drug
16,UKMYC5,0.120,Delamanid
17,UKMYC6,0.120,Delamanid
55,MGIT,0.060,Delamanid
87,7H11,0.016,Delamanid


In [38]:
cc_df_who

,Medium,Value,Drug
0,UKMYC5,0.100,Isoniazid
1,UKMYC6,0.100,Isoniazid
2,UKMYC5,0.500,Rifampicin
3,UKMYC6,0.500,Rifampicin
4,UKMYC5,4.000,Ethambutol
...,...,...,...
83,MYCOTB,0.120,Rifabutin
84,MYCOTB,0.120,Isoniazid
85,7H10,1.000,Clofazimine
86,BMD,0.500,Clofazimine


In [76]:
cc_df_who = pd.read_csv("/home/sak0914/who-analysis/data/drug_CC.csv")

cc_df_who_add = cc_lab[['antb', 'BMD', 'nsw_sensititre']].melt(id_vars=['antb']).dropna()
cc_df_who_add['Drug'] = cc_df_who_add['antb'].str.lower().str.capitalize()
cc_df_who_add.rename(columns={'variable': 'Medium', 'value': 'Value'}, inplace=True)
cc_df_who_add['Medium'] = cc_df_who_add['Medium'].replace('nsw_sensititre', 'NSW_sensititre_plate')

cc_df_who_final = pd.concat([cc_df_who, cc_df_who_add[cc_df_who.columns]]).query("Drug in @cc_df_who.Drug")

In [101]:
cc_df_who_final

,Medium,Value,Drug
0,UKMYC5,0.1,Isoniazid
1,UKMYC6,0.1,Isoniazid
2,UKMYC5,0.5,Rifampicin
3,UKMYC6,0.5,Rifampicin
4,UKMYC5,4.0,Ethambutol
...,...,...,...
29,NSW_sensititre_plate,0.1,Isoniazid
33,NSW_sensititre_plate,1.0,Moxifloxacin
38,NSW_sensititre_plate,0.5,Rifabutin
39,NSW_sensititre_plate,0.5,Rifampicin


In [79]:
cc_df_who_final.Medium.value_counts()

Medium
MGIT                    17
UKMYC5                  13
UKMYC6                  13
7H10                    13
LJ                      12
BMD                     12
7H11                    10
MYCOTB                   9
NSW_sensititre_plate     8
Name: count, dtype: int64

In [57]:
# cc_lab = pd.read_csv("//n/data1/hms/dbmi/farhat/rollingDB/metadata/MIC/critical_concentrations_all.csv")

# MIC-ML Data: Need to split bounds and normalize

## To split bounds, get the tested concentrations for each individual dataset

In [21]:
def process_bounds_MICML_data(df, drug):

    if drug not in df.columns:
        return pd.DataFrame()
    
    df_single_drug = df.loc[~pd.isnull(df[drug])]
    new_dfs = []
    
    for db in df_single_drug["DB_OF_ORIGIN"].unique():
        
        df_single_db = df_single_drug.query("DB_OF_ORIGIN==@db").reset_index(drop=True)
        db_bounds = list(np.sort(np.unique(df_single_db[drug]))) # make it a list so you can use the index method

        # check that each study only used one medium for each drug
        assert df_single_db['MEDIA'].nunique() == 1

        print(f"Study: {db}, Medium: {df_single_db['MEDIA'].unique()[0]}, Breakpoints: {db_bounds}")
        
        for i, row in df_single_db.iterrows():
                            
            # set the midpoint to the second smallest value because the MIC can't actually be 0
            if row[drug] == 0:
                midpoint_idx = 1
            else:
                midpoint_idx = db_bounds.index(row[drug])
            
            # the recorded value is the upper bound
            new_high = db_bounds[midpoint_idx]
            
            # if the upper bound is the smallest concentration, the lower bound should be 0
            if midpoint_idx == 0:
                new_low = 0 #db_bounds[midpoint_idx]
            # the lower bound should be 1 concentration below the upper bound
            else:
                new_low = db_bounds[midpoint_idx-1]

            # these MICs have already been normalized, so add the NORM suffix to the names
            df_single_db.loc[i, [f"{drug}_lower_bound_NORM", f"{drug}_upper_bound_NORM", f"{drug}_midpoint_NORM"]] = [new_low, new_high, np.mean([new_low, new_high])]
                
        new_dfs.append(df_single_db)
    
    df_final = pd.concat(new_dfs, axis=0)
    assert len(df_final) == len(df_single_drug)
    assert len(set(df_final["ROLLINGDB_ID"]).symmetric_difference(df_single_drug["ROLLINGDB_ID"])) == 0

    if drug in ['BDQ', 'DLM']:
        df_final[f'{drug}_MEDIA_NORM'] = '7H11'
    else:
        df_final[f'{drug}_MEDIA_NORM'] = '7H10'

    cols = ['ID', 'ROLLINGDB_ID', 'ISOLATION_LOCATION', 'DB_OF_ORIGIN', 'TESTING_LOCATION', 'MEDIA', f'{drug}_MEDIA_NORM'] + [f"{drug}_quality", f"{drug}_lower_bound_NORM", f"{drug}_midpoint_NORM", f"{drug}_upper_bound_NORM"]
    keep_cols = list(set(cols).intersection(df_final.columns))
    return df_final[keep_cols]

In [23]:
drug = 'RIF'

df_MIC_ML_single_drug = process_bounds_MICML_data(df_MIC_ML, drug)

Study: Cox, Medium: MGIT, Breakpoints: [0.062, 0.125, 0.25, 0.5, 1.0, 5.0, 10.0]
Study: EXIT-RIF, Medium: MGIT, Breakpoints: [0.25, 1.5, 3.75, 7.5, 11.25, 37.5, 50.0]
Study: JATA-KIT, Medium: MGIT, Breakpoints: [0.008, 0.015, 0.03, 0.06, 0.125, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 24.0, 32.0]
Study: JATA-Mongolia, Medium: MGIT, Breakpoints: [1.0, 2.0, 8.0, 32.0]
Study: Kreiswirth, Medium: 7H10, Breakpoints: [0.03, 0.06, 0.12, 0.125, 0.25]
Study: NICD, Medium: BMD, Breakpoints: [0.12, 0.48, 1.0, 2.0, 4.0, 8.0, 16.0]
Study: NSW, Medium: NSW_sensititre_plate, Breakpoints: [0.06, 0.25, 0.5, 16.0]
Study: Wadsworth, Medium: UKMYC, Breakpoints: [0.06, 0.12, 0.25, 0.5, 0.62, 1.0, 2.0, 4.0, 8.0, 16.0]
Study: CASS, Medium: MYCOTB, Breakpoints: [0.03, 0.06, 0.125, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 4.0, 8.0]


In [24]:
df_MIC_ML_single_drug

,TESTING_LOCATION,ISOLATION_LOCATION,ROLLINGDB_ID,RIF_midpoint_NORM,DB_OF_ORIGIN,RIF_MEDIA_NORM,RIF_upper_bound_NORM,ID,MEDIA,RIF_lower_bound_NORM
0,Not specified,NaN,20542,0.7500,Cox,7H10,1.000,20542,MGIT,0.500
1,Not specified,NaN,21849,0.0310,Cox,7H10,0.062,21849,MGIT,0.000
2,Not specified,NaN,23768,3.0000,Cox,7H10,5.000,23768,MGIT,1.000
3,Not specified,NaN,24304,0.1875,Cox,7H10,0.250,24304,MGIT,0.125
4,Not specified,NaN,26243,0.1875,Cox,7H10,0.250,26243,MGIT,0.125
...,...,...,...,...,...,...,...,...,...,...
399,Not specified,NaN,C501,0.0150,CASS,7H10,0.030,C501,MYCOTB,0.000
400,Not specified,NaN,C502,0.0150,CASS,7H10,0.030,C502,MYCOTB,0.000
401,Not specified,NaN,C503,6.0000,CASS,7H10,8.000,C503,MYCOTB,4.000
402,Not specified,NaN,C505,0.0150,CASS,7H10,0.030,C505,MYCOTB,0.000


In [25]:
cc_df

,antb,m7h10,m7h11,lj,mgit960,ABBR,UKMYC
0,AMIKACIN,4.0,NaN,30.0,1.00,AMI,1.00
1,CAPREOMYCIN,4.0,10.000,40.0,2.50,CAP,NaN
2,CIPROFLOXACIN,NaN,NaN,NaN,NaN,CIP,NaN
3,CYCLOSERINE,NaN,NaN,30.0,NaN,CYS,NaN
4,ETHAMBUTOL,5.0,7.500,2.0,5.00,EMB,4.00
5,ETHIONAMIDE,5.0,10.000,40.0,5.00,ETA,4.00
6,GATIFLOXACIN,1.0,NaN,NaN,NaN,GATI,NaN
7,ISONIAZID,0.2,0.200,0.2,0.10,INH,0.10
8,KANAMYCIN,5.0,6.000,30.0,2.50,KAN,4.00
9,LEVOFLOXACIN,1.0,NaN,NaN,1.50,LEVO,1.00
